Loads the Kaggle LinkedIn postings CSV from the raw_files volume
into jobmarket.bronze.job_postings_kaggle. Run once for backfill.
Reads:  /Volumes/jobmarket/bronze/raw_files/kaggle_linkedin/2026-07-06/postings.csv
Writes: jobmarket.bronze.job_postings_kaggle

In [0]:
path = "/Volumes/jobmarket/bronze/raw_files/kaggle_linkedin/2026-07-06/postings.csv"

df = (spark.read
      .option("header", True)
      .option("multiLine", True)
      .option("escape", '"')
      .csv(path))

df.printSchema()

In [0]:
print("Rows:", df.count())

In [0]:
display(df.select("title", "company_name", "location", "min_salary", "max_salary", "pay_period").limit(10))

In [0]:
from pyspark.sql import functions as F

cols = ["title", "company_name", "location", "description",
        "min_salary", "max_salary", "pay_period"]

df.select([
    (F.count(F.when(F.col(c).isNull() | (F.col(c) == ""), c)) / F.count("*") * 100)
    .alias(c + "_null_pct")
    for c in cols
]).display()

In [0]:
bronze = (df
    .withColumn("source", F.lit("kaggle_backfill"))
    .withColumn("ingest_ts", F.current_timestamp())
    .withColumn("ingest_date", F.current_date()))

(bronze.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("jobmarket.bronze.job_postings_kaggle"))

In [0]:
spark.table("jobmarket.bronze.job_postings_kaggle").count()